In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/visa_estimator_feature_engineered.csv")
df.head()


Mounted at /content/drive


,education_level,experience_years,annual_salary_usd,company_size,age,country_score,job_skill_score,previous_visas,application_score,visa_approval_score,season_index,experience_band,salary_per_experience
0,4,15,52939,1196,42,0.501252,0.313991,5,0.588091,100.0,3,4,3308.687500
1,5,13,80601,5168,39,0.902772,0.994638,1,0.896613,100.0,4,4,5757.214286
2,3,9,48101,7874,21,0.925827,0.644535,4,0.925796,100.0,1,3,4810.100000
3,5,6,92710,9932,51,0.724348,0.633114,1,0.605607,100.0,3,3,13244.285714
4,5,12,41514,4099,54,0.546535,0.734640,4,0.856800,100.0,3,4,3193.384615


In [2]:
from sklearn.model_selection import train_test_split

X = df.drop("visa_approval_score", axis=1)
y = df["visa_approval_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [4]:
lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)


In [5]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)


In [6]:
gb = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
gb.fit(X_train, y_train)

gb_pred = gb.predict(X_test)


In [7]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "Gradient Boosting"],
    "MAE": [
        evaluate(y_test, lr_pred)[0],
        evaluate(y_test, rf_pred)[0],
        evaluate(y_test, gb_pred)[0]
    ],
    "RMSE": [
        evaluate(y_test, lr_pred)[1],
        evaluate(y_test, rf_pred)[1],
        evaluate(y_test, gb_pred)[1]
    ],
    "R2 Score": [
        evaluate(y_test, lr_pred)[2],
        evaluate(y_test, rf_pred)[2],
        evaluate(y_test, gb_pred)[2]
    ]
})

results


,Model,MAE,RMSE,R2 Score
0,Linear Regression,0.644873,1.685024,0.116361
1,Random Forest,0.269988,1.182088,0.565127
2,Gradient Boosting,0.366923,1.261876,0.504440


In [8]:
train_meta = np.column_stack([
    lr.predict(X_train),
    rf.predict(X_train),
    gb.predict(X_train)
])

test_meta = np.column_stack([
    lr.predict(X_test),
    rf.predict(X_test),
    gb.predict(X_test)
])


In [9]:
meta_model = LinearRegression()
meta_model.fit(train_meta, y_train)

stacked_pred = meta_model.predict(test_meta)


In [10]:
stack_mae, stack_rmse, stack_r2 = evaluate(y_test, stacked_pred)

stack_results = pd.DataFrame({
    "Model": ["Stacked Ensemble"],
    "MAE": [stack_mae],
    "RMSE": [stack_rmse],
    "R2 Score": [stack_r2]
})

stack_results


,Model,MAE,RMSE,R2 Score
0,Stacked Ensemble,0.326873,1.159945,0.581267


In [11]:
import numpy as np

train_meta = np.column_stack([
    rf.predict(X_train),
    gb.predict(X_train)
])

test_meta = np.column_stack([
    rf.predict(X_test),
    gb.predict(X_test)
])


In [12]:
from sklearn.linear_model import Ridge

meta_model = Ridge(alpha=1.0)
meta_model.fit(train_meta, y_train)

stacked_pred = meta_model.predict(test_meta)


In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, stacked_pred)
rmse = np.sqrt(mean_squared_error(y_test, stacked_pred))
r2 = r2_score(y_test, stacked_pred)

mae, rmse, r2


(0.3237688279790199, np.float64(1.160532431427469), 0.5808425312623933)

In [15]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


In [16]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

lgbm = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)


In [17]:
xgb.fit(X_train, y_train)
lgbm.fit(X_train, y_train)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000556 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1608
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 12
[LightGBM] [Info] Start training from score 99.756925


LGBMRegressor(learning_rate=0.05, n_estimators=300, random_state=42)

In [18]:
import numpy as np

train_meta_ext = np.column_stack([
    rf.predict(X_train),
    gb.predict(X_train),
    xgb.predict(X_train),
    lgbm.predict(X_train)
])

test_meta_ext = np.column_stack([
    rf.predict(X_test),
    gb.predict(X_test),
    xgb.predict(X_test),
    lgbm.predict(X_test)
])


In [19]:
from sklearn.linear_model import Ridge

meta_model_ext = Ridge(alpha=1.0)
meta_model_ext.fit(train_meta_ext, y_train)

stacked_pred_ext = meta_model_ext.predict(test_meta_ext)


In [20]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_ext = mean_absolute_error(y_test, stacked_pred_ext)
rmse_ext = np.sqrt(mean_squared_error(y_test, stacked_pred_ext))
r2_ext = r2_score(y_test, stacked_pred_ext)

mae_ext, rmse_ext, r2_ext


(0.2897010995050973, np.float64(1.0729580611868474), 0.6417154011321857)

In [21]:
mape = np.mean(np.abs((y_test - stacked_pred_ext) / y_test)) * 100
accuracy_percentage = 100 - mape

accuracy_percentage


np.float64(99.69352222487466)